# 13 — Interaction Features: Does `pre_quality × team_style` Unlock Better Predictions?

From nb12: team context adds Δ R²=+0.023 on average in linear form. Small.

**Hypothesis**: the effect isn’t linear. It’s not *“high PENETRATION = better Box Threat”*. It’s *“a good dribbler going to a high-PENETRATION team sees their Box Threat explode.”* That’s an interaction: `pre_Dribbling × to_PENETRATION`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'
raw = pd.read_parquet(f'{V1}/model_df.parquet')

In [ ]:
df = raw[raw['position'] != 'Goalkeeper'].copy()
team_cols = [c for c in df.columns if c.startswith('from_q_proj_') or c.startswith('to_q_') or c.startswith('delta_team_')]
df = df[~df[team_cols].isna().any(axis=1)].copy()

ALL_PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]
TEAM_ALL = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
            'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']
POSITIONS = sorted(df['position'].unique())

position_qualities = {}
for pos in POSITIONS:
    mask = df['position'] == pos
    position_qualities[pos] = [q for q in ALL_PLAYER_QUALITIES if df.loc[mask, f'pre_{q}'].isna().mean() < 0.50]

df['style_distance'] = np.sqrt(((df[[f'to_q_{q}' for q in TEAM_ALL]].values -
                                  df[[f'from_q_proj_{q}' for q in TEAM_ALL]].values) ** 2).sum(axis=1))

train_mask = df['to_season'].isin([2020, 2021, 2022, 2023])
test_mask = df['to_season'] == 2024
print(f'{len(df):,} transfers | Train: {train_mask.sum():,} | Test: {test_mask.sum():,}')

## 1. Generate Interaction Features

`pre_quality × to_team_style` for all combinations. Captures: *“given who you are AND where you’re going, what happens?”*

In [ ]:
# Generate interaction columns
interaction_cols_by_pos = {}

for pos in POSITIONS:
    qualities = position_qualities[pos]
    int_cols = []
    for q in qualities:
        for t in TEAM_ALL:
            col_name = f'int_pre{q}_x_to{t}'
            if col_name not in df.columns:
                df[col_name] = df[f'pre_{q}'] * df[f'to_q_{t}']
            int_cols.append(col_name)
    interaction_cols_by_pos[pos] = int_cols
    print(f'{pos:20s}: {len(qualities)} pre × 7 team = {len(int_cols)} interactions')

print(f'\nTotal columns in df: {len(df.columns)}')

## 2. Linear vs Interaction Models

Per (position × quality): Lasso with linear features only vs Lasso with linear + interactions.

In [ ]:
ALPHAS = np.logspace(-3, 1, 20)
results = []
fitted_models = {}

for pos in POSITIONS:
    qualities = position_qualities[pos]
    pre_cols = [f'pre_{q}' for q in qualities]
    team_features = ([f'from_q_proj_{q}' for q in TEAM_ALL] +
                     [f'to_q_{q}' for q in TEAM_ALL] +
                     ['style_distance', 'pre_minutes'])
    linear_cols = pre_cols + team_features
    interact_cols = linear_cols + interaction_cols_by_pos[pos]

    pos_df = df[df['position'] == pos]

    for q in qualities:
        target = f'post_{q}'
        needed = interact_cols + [target]  # superset
        clean = pos_df[needed].dropna()
        train_idx = clean.index[clean.index.isin(df[train_mask].index)]
        test_idx = clean.index[clean.index.isin(df[test_mask].index)]

        if len(train_idx) < 20 or len(test_idx) < 5:
            continue

        y_train = clean.loc[train_idx, target].values
        y_test = clean.loc[test_idx, target].values

        # Linear only
        X_tr_lin = clean.loc[train_idx, linear_cols].values
        X_te_lin = clean.loc[test_idx, linear_cols].values
        m_lin = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X_tr_lin, y_train)
        r2_lin = r2_score(y_test, m_lin.predict(X_te_lin))

        # With interactions
        X_tr_int = clean.loc[train_idx, interact_cols].values
        X_te_int = clean.loc[test_idx, interact_cols].values
        m_int = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X_tr_int, y_train)
        y_pred_int = m_int.predict(X_te_int)
        r2_int = r2_score(y_test, y_pred_int)

        # Interaction feature analysis
        coefs_int = pd.Series(m_int.coef_, index=interact_cols)
        int_only = coefs_int[interaction_cols_by_pos[pos]]
        int_nonzero = int_only[int_only != 0].sort_values(key=abs, ascending=False)

        top_int = ''
        if len(int_nonzero) > 0:
            top3 = int_nonzero.head(3)
            top_int = ', '.join([f'{n.replace("int_pre","").replace("_x_to"," × ")}({v:+.3f})'
                                 for n, v in top3.items()])

        results.append({
            'position': pos, 'quality': q,
            'r2_linear': r2_lin, 'r2_interact': r2_int,
            'delta_r2': r2_int - r2_lin,
            'n_interactions_kept': (int_nonzero != 0).sum() if len(int_nonzero) > 0 else 0,
            'n_total_kept': (coefs_int != 0).sum(),
            'top_interactions': top_int,
            'n_train': len(train_idx), 'n_test': len(test_idx),
        })

        fitted_models[(pos, q)] = {
            'model_int': m_int, 'cols': interact_cols,
            'y_test': y_test, 'y_pred': y_pred_int, 'coefs': coefs_int,
        }

    print(f'  {pos}: done')

res_df = pd.DataFrame(results).sort_values('delta_r2', ascending=False)
print(f'\n{len(res_df)} models. Mean \u0394 R\u00b2 = {res_df["delta_r2"].mean():.4f}')
print(f'Combos where interactions help (\u0394 > 0): {(res_df["delta_r2"] > 0).sum()} / {len(res_df)}')
print(f'Combos where interactions help a lot (\u0394 > 0.03): {(res_df["delta_r2"] > 0.03).sum()}')

## 3. Where Interactions Help Most

In [ ]:
# Summary by position
pos_comp = res_df.groupby('position')[['r2_linear', 'r2_interact', 'delta_r2']].mean().round(4)
print('MEAN R\u00b2: LINEAR vs INTERACTIONS')
print(pos_comp.to_string())
print(f'\nOverall: linear={res_df["r2_linear"].mean():.4f}, interact={res_df["r2_interact"].mean():.4f}, \u0394={res_df["delta_r2"].mean():.4f}')

In [ ]:
# Heatmap: delta R² (interact - linear)
pivot_delta = res_df.pivot_table(index='quality', columns='position', values='delta_r2')

fig = px.imshow(
    pivot_delta.round(3),
    text_auto='.3f',
    color_continuous_scale='RdBu',
    zmin=-0.1, zmax=0.1,
    labels=dict(x='Position', y='Quality', color='\u0394 R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title='\u0394 R\u00b2 (Interactions \u2212 Linear) \u2014 Where Interactions Unlock Power',
    height=600, width=650,
    margin=dict(t=50, b=30, l=170, r=30),
)
fig.show()

In [ ]:
# Top combos where interactions helped most
top = res_df.nlargest(15, 'delta_r2').copy()
top['label'] = top['position'].str[:3] + ' \u2014 ' + top['quality']
top = top.sort_values('delta_r2', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=top['label'], x=top['r2_linear'], name='Linear only',
                      orientation='h', marker_color='#636EFA'))
fig.add_trace(go.Bar(y=top['label'], x=top['r2_interact'], name='+ Interactions',
                      orientation='h', marker_color='#00CC96'))
fig.update_layout(
    title='Top 15: Where Interactions Add Most Value',
    barmode='group', height=500, width=750,
    margin=dict(t=50, b=30, l=160, r=30),
    xaxis_title='R\u00b2', legend=dict(x=0.6, y=0.05),
)
fig.show()

## 4. The Interaction Stories

Which `pre_quality × to_team_style` interactions survived Lasso? These are the non-obvious, conditional effects.

In [ ]:
# Top stories: combos with biggest delta AND meaningful interactions
stories = res_df[(res_df['delta_r2'] > 0.01) & (res_df['n_interactions_kept'] > 0)].copy()
stories = stories.sort_values('delta_r2', ascending=False)
print(f'{len(stories)} combos where interactions added \u0394 R\u00b2 > 0.01\n')
print(stories[['position', 'quality', 'r2_linear', 'r2_interact', 'delta_r2',
               'n_interactions_kept', 'top_interactions']].to_string(index=False))

In [ ]:
# Coefficient bar charts for top 6 stories
top_stories = stories.head(6)

for _, row in top_stories.iterrows():
    key = (row['position'], row['quality'])
    info = fitted_models[key]
    coefs = info['coefs']
    nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)

    # Shorten names
    short = nonzero.head(12).copy()
    short.index = (short.index
        .str.replace('int_pre', '\u2731 ', regex=False)
        .str.replace('_x_to', ' \u00d7 ', regex=False)
        .str.replace('from_q_proj_', 'from ', regex=False)
        .str.replace('to_q_', 'to ', regex=False)
        .str.replace('pre_', 'pre ', regex=False))

    # Color: interactions in orange, linear in blue
    colors = ['#FF6B35' if '\u2731' in str(idx) else '#636EFA' for idx in short.index]

    fig = go.Figure(go.Bar(
        x=short.values, y=short.index,
        orientation='h', marker_color=colors,
    ))
    fig.update_layout(
        title=f"{row['position']} \u2014 post {row['quality']} (R\u00b2={row['r2_interact']:.3f}, \u0394={row['delta_r2']:+.3f})",
        height=max(300, len(short) * 30),
        width=650,
        margin=dict(t=40, b=30, l=220, r=30),
        xaxis_title='Coefficient',
        yaxis=dict(autorange='reversed'),
    )
    fig.add_annotation(text='\U0001f7e0 = interaction feature', xref='paper', yref='paper',
                       x=0.95, y=-0.08, showarrow=False, font=dict(size=10))
    fig.show()

## 5. Summary

In [ ]:
# Full comparison table
print('FULL RESULTS (sorted by \u0394 R\u00b2)')
top20 = res_df.head(20)
print(top20[['position', 'quality', 'r2_linear', 'r2_interact', 'delta_r2',
             'n_interactions_kept', 'top_interactions']].to_string(index=False))

In [ ]:
# Final numbers
mean_lin = res_df['r2_linear'].mean()
mean_int = res_df['r2_interact'].mean()
pct_improved = (res_df['delta_r2'] > 0).mean() * 100
n_strong_delta = (res_df['delta_r2'] > 0.03).sum()
best = res_df.iloc[0]

print('CONCLUSIONS')
print('=' * 70)
print(f'\nMean R\u00b2: linear={mean_lin:.4f}, +interactions={mean_int:.4f}, \u0394={mean_int-mean_lin:+.4f}')
print(f'Interactions improved {pct_improved:.0f}% of quality-position combos')
print(f'{n_strong_delta} combos with \u0394 R\u00b2 > 0.03')
print(f'\nBiggest win: {best["position"]} \u2014 {best["quality"]}'
      f' (linear={best["r2_linear"]:.3f} \u2192 interact={best["r2_interact"]:.3f}, \u0394={best["delta_r2"]:+.3f})')
print(f'  Top interactions: {best["top_interactions"]}')